In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


# 1. Define input and output directories
input_folder = Path("results/step_02")
output_folder = Path("results/step_03")

output_folder.mkdir(
    parents=True,
    exist_ok=True
)


# 2. Load the results generated by Code 2
mcdm_df = pd.read_csv(
    input_folder / "aggregated_criteria.csv"
)

normalized_df = pd.read_csv(
    input_folder / "normalized_criteria.csv",
    index_col="Reference"
)

weights_df = pd.read_csv(
    input_folder / "critic_weights.csv"
)

criteria = [
    "Average_Quantity",
    "Volatility",
    "Average_Unit_Margin",
    "Total_Cost"
]

# Align rows and criterion weights explicitly.
normalized_df = normalized_df.loc[
    mcdm_df["Reference"],
    criteria
]

weights = (
    weights_df
    .set_index("Criterion")
    .loc[criteria, "Weight"]
    .to_numpy(dtype=float)
)

normalized_values = normalized_df.to_numpy(
    dtype=float
)

assert np.isclose(weights.sum(), 1.0)
assert np.isfinite(normalized_values).all()


# 3. Define the TOPSIS scoring function
def calculate_topsis(values, criterion_weights):

    # All criteria are already preference-oriented:
    # higher normalized values indicate higher priority.
    weighted_values = (
        np.asarray(values, dtype=float)
        * np.asarray(criterion_weights, dtype=float)
    )

    # Identify the positive and negative ideal solutions.
    positive_ideal = weighted_values.max(axis=0)
    negative_ideal = weighted_values.min(axis=0)

    # Calculate Euclidean distances to both ideal solutions.
    distance_positive = np.linalg.norm(
        weighted_values - positive_ideal,
        axis=1
    )

    distance_negative = np.linalg.norm(
        weighted_values - negative_ideal,
        axis=1
    )

    total_distance = (
        distance_positive + distance_negative
    )

    # Assign 0.5 when all distances are zero.
    closeness = np.divide(
        distance_negative,
        total_distance,
        out=np.full(len(weighted_values), 0.5),
        where=total_distance > 1e-12
    )

    return (
        closeness,
        distance_positive,
        distance_negative
    )


# 4. Calculate Weighted Sum Model scores
mcdm_df["WSM_Score"] = (
    normalized_values @ weights
)


# 5. Calculate classical TOPSIS scores
(
    topsis_scores,
    topsis_distance_positive,
    topsis_distance_negative
) = calculate_topsis(
    normalized_values,
    weights
)

mcdm_df["TOPSIS_Score"] = topsis_scores

mcdm_df["TOPSIS_Distance_Positive"] = (
    topsis_distance_positive
)

mcdm_df["TOPSIS_Distance_Negative"] = (
    topsis_distance_negative
)


# 6. Define the fuzzification settings
FUZZY_MARGIN = 0.05

# False: literal symmetric fuzzification followed by centroid.
# True: additionally clip the triangle bounds to [0, 1].
# Clipping is a separate convention not specified in the article.
CLIP_FUZZY_BOUNDS = False


# 7. Construct triangular fuzzy numbers
# Each normalized value x becomes (x - margin, x, x + margin).
fuzzy_values = np.stack(
    [
        normalized_values - FUZZY_MARGIN,
        normalized_values,
        normalized_values + FUZZY_MARGIN
    ],
    axis=-1
)

if CLIP_FUZZY_BOUNDS:
    fuzzy_values = np.clip(
        fuzzy_values,
        0.0,
        1.0
    )


# 8. Defuzzify using the centroid
# Without clipping, the centroid is exactly x:
# ((x - margin) + x + (x + margin)) / 3 = x.
centroid_values = fuzzy_values.mean(
    axis=-1
)


# 9. Apply TOPSIS to the defuzzified matrix
# This implements the centroid-based procedure described
# in the article, rather than fuzzy-distance TOPSIS.
(
    fuzzy_topsis_scores,
    fuzzy_distance_positive,
    fuzzy_distance_negative
) = calculate_topsis(
    centroid_values,
    weights
)

mcdm_df["Fuzzy_TOPSIS_Score"] = (
    fuzzy_topsis_scores
)

mcdm_df["Fuzzy_TOPSIS_Distance_Positive"] = (
    fuzzy_distance_positive
)

mcdm_df["Fuzzy_TOPSIS_Distance_Negative"] = (
    fuzzy_distance_negative
)


# 10. Rank the parts under each method
methods = [
    "WSM",
    "TOPSIS",
    "Fuzzy_TOPSIS"
]

for method in methods:

    score_column = f"{method}_Score"
    rank_column = f"{method}_Rank"

    mcdm_df[rank_column] = (
        mcdm_df[score_column]
        .rank(
            ascending=False,
            method="min"
        )
        .astype(int)
    )


# 11. Compare the rankings using Spearman and Kendall correlations
rank_columns = [
    "WSM_Rank",
    "TOPSIS_Rank",
    "Fuzzy_TOPSIS_Rank"
]

spearman_correlation = (
    mcdm_df[rank_columns]
    .corr(method="spearman")
)

kendall_correlation = (
    mcdm_df[rank_columns]
    .corr(method="kendall")
)


# 12. Select the top 10 parts according to Fuzzy TOPSIS
top_10 = (
    mcdm_df
    .sort_values(
        by=["Fuzzy_TOPSIS_Score", "Reference"],
        ascending=[False, True]
    )
    .head(10)
)


# 13. Check the calculated scores
for method in methods:

    scores = mcdm_df[
        f"{method}_Score"
    ].to_numpy()

    assert np.isfinite(scores).all()

    assert (
        (scores >= -1e-12)
        & (scores <= 1 + 1e-12)
    ).all()

maximum_difference = np.max(
    np.abs(
        mcdm_df["TOPSIS_Score"]
        - mcdm_df["Fuzzy_TOPSIS_Score"]
    )
)

if not CLIP_FUZZY_BOUNDS:

    np.testing.assert_allclose(
        mcdm_df["TOPSIS_Score"],
        mcdm_df["Fuzzy_TOPSIS_Score"],
        rtol=0,
        atol=1e-12
    )


# 14. Display the results
display_columns = [
    "Reference",
    "WSM_Score",
    "TOPSIS_Score",
    "Fuzzy_TOPSIS_Score",
    "WSM_Rank",
    "TOPSIS_Rank",
    "Fuzzy_TOPSIS_Rank"
]

print("Top 10 parts ranked by Fuzzy TOPSIS:")
print(
    top_10[display_columns]
    .round(6)
    .to_string(index=False)
)

print("\nSpearman rank correlations:")
print(spearman_correlation.round(6))

print("\nKendall rank correlations:")
print(kendall_correlation.round(6))

print(
    "\nMaximum absolute difference between "
    "TOPSIS and Fuzzy TOPSIS:",
    maximum_difference
)

print(
    "\nFuzzy triangle bounds clipped:",
    CLIP_FUZZY_BOUNDS
)


# 15. Save the results
mcdm_df.to_csv(
    output_folder / "mcdm_scores_and_rankings.csv",
    index=False
)

top_10.to_csv(
    output_folder / "top_10_parts.csv",
    index=False
)

spearman_correlation.to_csv(
    output_folder / "spearman_rank_correlations.csv",
    index=True
)

kendall_correlation.to_csv(
    output_folder / "kendall_rank_correlations.csv",
    index=True
)

# Record the selected fuzzification convention.
settings_df = pd.DataFrame([{
    "Fuzzy_Margin": FUZZY_MARGIN,
    "Clip_Fuzzy_Bounds": CLIP_FUZZY_BOUNDS,
    "Maximum_TOPSIS_Fuzzy_Difference": maximum_difference
}])

settings_df.to_csv(
    output_folder / "fuzzification_settings.csv",
    index=False
)

print(f"\nResults saved in: {output_folder}")